In [27]:
import pandas as pd
smh = pd.read_excel("SMH_asof_20260827.xlsx",skiprows=2)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [28]:
smh

,Number,Ticker,Holding Name,Identifier (FIGI),Shares,Asset Class,Market Value (US$),Notional Value,% of Net Assets
0,1,NVDA,Nvidia Corp,BBG000BBJQV0,"70,357,684",Stock,"$16,040,144,798.32",--,22.89%
1,2,TSM,Taiwan Semiconductor Manufacturing Co L,BBG000BD8ZK0,"15,640,925",Stock,"$6,683,367,252.50",--,9.54%
2,3,AVGO,Broadcom Inc,BBG00KHY5S69,"11,204,001",Stock,"$4,162,734,531.54",--,5.94%
3,4,AMD,Advanced Micro Devices Inc,BBG000BBQCY0,"7,792,382",Stock,"$3,714,394,727.94",--,5.30%
4,5,MU,Micron Technology Inc,BBG000C5Z1S3,"3,952,614",Stock,"$3,697,235,609.46",--,5.28%
5,6,ASML,Asml Holding Nv,BBG000K6MRN4,"2,032,785",Stock,"$3,526,902,302.85",--,5.03%
6,7,LRCX,Lam Research Corp,BBG000BNFLM9,"9,857,473",Stock,"$3,140,393,748.34",--,4.48%
7,8,AMAT,Applied Materials Inc,BBG000BBPFB9,"6,383,665",Stock,"$3,079,224,649.40",--,4.39%
8,9,MRVL,Marvell Technology Inc,BBG00ZXBJ153,"12,560,884",Stock,"$3,032,825,441.80",--,4.33%
9,10,ADI,Analog Devices Inc,BBG000BB6G37,"8,079,919",Stock,"$3,026,091,263.88",--,4.32%


In [29]:
smh.columns


Index(['Number', 'Ticker', 'Holding Name', 'Identifier (FIGI)', 'Shares',
       'Asset Class', 'Market Value (US$)', 'Notional Value',
       '% of Net Assets'],
      dtype='object')

In [30]:
df = smh[smh["Asset Class"]== "Stock"].copy()

In [31]:
df["weight"] = df["% of Net Assets"].str.replace("%","" ).str.strip().astype(float)

In [32]:
df["market value"]= (df["Market Value (US$)"].str.replace("$","").str.replace(",","").str.strip().astype(float))

In [33]:
df

,Number,Ticker,Holding Name,Identifier (FIGI),Shares,Asset Class,Market Value (US$),Notional Value,% of Net Assets,weight,market value
0,1,NVDA,Nvidia Corp,BBG000BBJQV0,"70,357,684",Stock,"$16,040,144,798.32",--,22.89%,22.89,1.604014e+10
1,2,TSM,Taiwan Semiconductor Manufacturing Co L,BBG000BD8ZK0,"15,640,925",Stock,"$6,683,367,252.50",--,9.54%,9.54,6.683367e+09
2,3,AVGO,Broadcom Inc,BBG00KHY5S69,"11,204,001",Stock,"$4,162,734,531.54",--,5.94%,5.94,4.162735e+09
3,4,AMD,Advanced Micro Devices Inc,BBG000BBQCY0,"7,792,382",Stock,"$3,714,394,727.94",--,5.30%,5.30,3.714395e+09
4,5,MU,Micron Technology Inc,BBG000C5Z1S3,"3,952,614",Stock,"$3,697,235,609.46",--,5.28%,5.28,3.697236e+09
5,6,ASML,Asml Holding Nv,BBG000K6MRN4,"2,032,785",Stock,"$3,526,902,302.85",--,5.03%,5.03,3.526902e+09
6,7,LRCX,Lam Research Corp,BBG000BNFLM9,"9,857,473",Stock,"$3,140,393,748.34",--,4.48%,4.48,3.140394e+09
7,8,AMAT,Applied Materials Inc,BBG000BBPFB9,"6,383,665",Stock,"$3,079,224,649.40",--,4.39%,4.39,3.079225e+09
8,9,MRVL,Marvell Technology Inc,BBG00ZXBJ153,"12,560,884",Stock,"$3,032,825,441.80",--,4.33%,4.33,3.032825e+09
9,10,ADI,Analog Devices Inc,BBG000BB6G37,"8,079,919",Stock,"$3,026,091,263.88",--,4.32%,4.32,3.026091e+09


In [34]:
df["shares"]= df["Shares"].str.replace(",","").astype(int)

In [35]:
df["weight"] = df["weight"] / df["weight"].sum() * 100

In [37]:
df.columns

Index(['Number', 'Ticker', 'Holding Name', 'Identifier (FIGI)', 'Shares',
       'Asset Class', 'Market Value (US$)', 'Notional Value',
       '% of Net Assets', 'weight', 'market value', 'shares'],
      dtype='object')

In [43]:
df = df[["Ticker", "Holding Name", "Identifier (FIGI)", "shares", "market value", "weight"]].reset_index(drop=True)

In [45]:
df["weight"].sum()

np.float64(99.99999999999999)

In [49]:
from parsers import parse_vaneck
smh = parse_vaneck("data/raw/SMH_asof_20260827.xlsx", "SMH")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [52]:
import pandas as pd
from parsers import parse_vaneck, parse_ishares, parse_spdr

smh  = parse_vaneck("data/raw/SMH_asof_20260827.xlsx", "SMH")
soxx = parse_ishares("data/raw/SOXX_holdings.csv", "SOXX")
xsd  = parse_spdr("data/raw/holdings-daily-us-en-xsd.xlsx", "XSD")

tum = pd.concat([smh, soxx, xsd], ignore_index=True)

print(tum.groupby("fund")["weight"].agg(["count", "sum"]))
print()
print("Kolonlar:", tum.columns.tolist())

      count    sum
fund              
SMH      25  100.0
SOXX     30  100.0
XSD      47  100.0

Kolonlar: ['ticker', 'name', 'figi', 'shares', 'market_value', 'weight', 'cusip', 'sector', 'location', 'fund', 'date']


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/var/folders/lm/bj5bvm9s06x80pq_k_dgl1780000gn/T/ipykernel_24563/3940503595.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tum = pd.concat([smh, soxx, xsd], ignore_index=True)


In [53]:
tum

,ticker,name,figi,shares,market_value,weight,cusip,sector,location,fund,date
0,NVDA,Nvidia Corp,BBG000BBJQV0,70357684,1.604014e+10,22.903742,None,None,None,SMH,2026-08-27
1,TSM,Taiwan Semiconductor Manufacturing Co L,BBG000BD8ZK0,15640925,6.683367e+09,9.545727,None,None,None,SMH,2026-08-27
2,AVGO,Broadcom Inc,BBG00KHY5S69,11204001,4.162735e+09,5.943566,None,None,None,SMH,2026-08-27
3,AMD,Advanced Micro Devices Inc,BBG000BBQCY0,7792382,3.714395e+09,5.303182,None,None,None,SMH,2026-08-27
4,MU,Micron Technology Inc,BBG000C5Z1S3,3952614,3.697236e+09,5.283170,None,None,None,SMH,2026-08-27
...,...,...,...,...,...,...,...,...,...,...,...
97,AMBQ,AMBIQ MICRO INC,None,344193,NaN,0.773209,023193105,None,None,XSD,2026-08-27
98,CEVA,CEVA INC,None,685601,NaN,0.728430,157210105,None,None,XSD,2026-08-27
99,MRAM,EVERSPIN TECHNOLOGIES INC,None,870627,NaN,0.568289,30041T104,None,None,XSD,2026-08-27
100,NVEC,NVE CORP,None,129606,NaN,0.518832,629445206,None,None,XSD,2026-08-27


In [54]:
def yogunlasma(df):
    d = df.sort_values("weight", ascending=False)
    w = d["weight"] / 100
    return pd.Series({
        "hisse_sayisi": len(d),
        "en_buyuk": d["weight"].iloc[0],
        "top_3": d["weight"].head(3).sum(),
        "top_10": d["weight"].head(10).sum(),
        "hhi": (w ** 2).sum() * 10000,
        "etkin_hisse": 1 / (w ** 2).sum(),
    })

print(tum.groupby("fund").apply(yogunlasma).round(2))

      hisse_sayisi  en_buyuk  top_3  top_10     hhi  etkin_hisse
fund                                                            
SMH           25.0     22.90  38.39   71.54  890.82        11.23
SOXX          30.0      9.47  26.03   61.52  513.04        19.49
XSD           47.0      3.67   9.76   28.74  236.12        42.35


/var/folders/lm/bj5bvm9s06x80pq_k_dgl1780000gn/T/ipykernel_24563/2048533112.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(tum.groupby("fund").apply(yogunlasma).round(2))


In [56]:
import yfinance as yf

tickerlar = sorted(tum["ticker"].unique())
print(len(tickerlar), "benzersiz hisse")
print(tickerlar)

61 benzersiz hisse
['ADI', 'ALAB', 'ALGM', 'AMAT', 'AMBA', 'AMBQ', 'AMD', 'AOSL', 'ARM', 'ASML', 'ASX', 'AVGO', 'CBRS', 'CDNS', 'CEVA', 'CRDO', 'CRUS', 'DIOD', 'ENTG', 'FSLR', 'INDI', 'INTC', 'IONQ', 'KLAC', 'KOPN', 'LRCX', 'LSCC', 'MCHP', 'MPWR', 'MRAM', 'MRVL', 'MTSI', 'MU', 'MXL', 'NVDA', 'NVEC', 'NVMI', 'NVTS', 'NXPI', 'OLED', 'ON', 'PENG', 'PI', 'POWI', 'QCOM', 'QRVO', 'RGTI', 'RMBS', 'SITM', 'SLAB', 'SMTC', 'SNPS', 'STM', 'SWKS', 'SYNA', 'TE', 'TER', 'TSM', 'TXN', 'UMC', 'WOLF']


In [57]:
import yfinance as yf

fiyat = yf.download(
    tickerlar + ["GC=F", "^GSPC", "BIL"],
    start="2025-08-01",
    end="2026-08-29",
    auto_adjust=True,
    progress=False,
)["Close"]

print("Boyut:", fiyat.shape)
print()
print("Eksik veri (gün sayısı):")
eksik = fiyat.isna().sum().sort_values(ascending=False)
print(eksik[eksik > 0])

Boyut: (271, 64)

Eksik veri (gün sayısı):
Ticker
CBRS     198
WOLF      41
ADI        1
OLED       1
QCOM       1
        ... 
LRCX       1
LSCC       1
MCHP       1
MPWR       1
^GSPC      1
Length: 64, dtype: int64


In [58]:
# Herkeste ortak eksik olan günler
tam_bos = fiyat.index[fiyat.isna().all(axis=1)]
print("Tamamen boş günler:", tam_bos)

# 62 ticker'da eksik olan gün hangisi
cok_eksik = fiyat.index[fiyat.isna().sum(axis=1) > 50]
print("50+ ticker'da eksik olan günler:", cok_eksik)


Tamamen boş günler: DatetimeIndex(['2026-08-28'], dtype='datetime64[ns]', name='Date', freq=None)
50+ ticker'da eksik olan günler: DatetimeIndex(['2026-08-28'], dtype='datetime64[ns]', name='Date', freq=None)


In [59]:
# Tamamen boş günleri at
fiyat = fiyat.dropna(how="all")

# Yetersiz geçmişi olanları işaretle (silme, ayır)
min_gun = len(fiyat) * 0.9
gecerli = fiyat.columns[fiyat.notna().sum() >= min_gun].tolist()
eksikli = [t for t in fiyat.columns if t not in gecerli]

print(f"{len(fiyat)} işlem günü")
print(f"Geçerli: {len(gecerli)} | Yetersiz veri: {eksikli}")

270 işlem günü
Geçerli: 62 | Yetersiz veri: ['CBRS', 'WOLF']


In [60]:
def donem_getirisi(fiyat_df, gun):
    """Son 'gun' işlem günündeki yüzde getiri."""
    p = fiyat_df.iloc[-gun:]
    return ((p.iloc[-1] / p.iloc[0]) - 1) * 100

getiri_3ay = donem_getirisi(fiyat[gecerli], 63)   # ~3 ay
getiri_1yil = donem_getirisi(fiyat[gecerli], 252) # ~1 yıl

print(getiri_3ay.sort_values(ascending=False).head(10).round(1))

Ticker
MRVL     17.8
PI       11.3
NVEC     10.3
NVDA      8.1
ASML      7.7
AMAT      7.3
ENTG      4.8
ASX       2.5
TSM       2.4
^GSPC     2.0
dtype: float64


In [61]:
def katki_analizi(tum_df, getiri_serisi, fon):
    d = tum_df[tum_df["fund"] == fon].copy()
    d["getiri"] = d["ticker"].map(getiri_serisi)

    # Veri olmayanları ayır
    veri_yok = d[d["getiri"].isna()]
    d = d.dropna(subset=["getiri"])

    # Katkı = ağırlık × getiri
    d["katki"] = (d["weight"] / 100) * d["getiri"]

    d = d.sort_values("katki", ascending=False)

    print(f"=== {fon} ===")
    print(f"Toplam katkı: {d['katki'].sum():.2f}%")
    if len(veri_yok):
        print(f"Hesaplanamayan: {veri_yok['ticker'].tolist()} "
              f"(ağırlık {veri_yok['weight'].sum():.2f}%)")
    print()
    print(d[["ticker", "weight", "getiri", "katki"]].head(8).round(2).to_string(index=False))
    print("...")
    print(d[["ticker", "weight", "getiri", "katki"]].tail(5).round(2).to_string(index=False))
    print()
    return d

for f in ["SMH", "SOXX", "XSD"]:
    katki_analizi(tum, getiri_3ay, f)

=== SMH ===
Toplam katkı: -2.96%

ticker  weight  getiri  katki
  NVDA   22.90    8.10   1.86
  MRVL    4.33   17.81   0.77
  ASML    5.03    7.72   0.39
  AMAT    4.39    7.29   0.32
   TSM    9.55    2.35   0.22
  LRCX    4.48    0.20   0.01
   TER    1.32   -0.60  -0.01
  SWKS    0.18  -13.47  -0.02
...
ticker  weight  getiri  katki
   AMD    5.30   -7.64  -0.41
   TXN    4.28  -12.36  -0.53
  INTC    3.90  -19.70  -0.77
  AVGO    5.94  -16.71  -0.99
  QCOM    3.82  -34.11  -1.30

=== SOXX ===
Toplam katkı: -6.62%

ticker  weight  getiri  katki
  MRVL    5.17   17.81   0.92
  NVDA    9.47    8.10   0.77
  AMAT    4.70    7.29   0.34
  ASML    2.45    7.72   0.19
   TSM    4.66    2.35   0.11
  ENTG    1.20    4.79   0.06
  CRDO    2.16    1.78   0.04
   ASX    1.28    2.45   0.03
...
ticker  weight  getiri  katki
   AMD    8.10   -7.64  -0.62
  NXPI    3.08  -29.48  -0.91
  QCOM    2.87  -34.11  -0.98
  INTC    5.10  -19.70  -1.00
  AVGO    7.29  -16.71  -1.22

=== XSD ===
Toplam ka

In [63]:
tum

,ticker,name,figi,shares,market_value,weight,cusip,sector,location,fund,date
0,NVDA,Nvidia Corp,BBG000BBJQV0,70357684,1.604014e+10,22.903742,None,None,None,SMH,2026-08-27
1,TSM,Taiwan Semiconductor Manufacturing Co L,BBG000BD8ZK0,15640925,6.683367e+09,9.545727,None,None,None,SMH,2026-08-27
2,AVGO,Broadcom Inc,BBG00KHY5S69,11204001,4.162735e+09,5.943566,None,None,None,SMH,2026-08-27
3,AMD,Advanced Micro Devices Inc,BBG000BBQCY0,7792382,3.714395e+09,5.303182,None,None,None,SMH,2026-08-27
4,MU,Micron Technology Inc,BBG000C5Z1S3,3952614,3.697236e+09,5.283170,None,None,None,SMH,2026-08-27
...,...,...,...,...,...,...,...,...,...,...,...
97,AMBQ,AMBIQ MICRO INC,None,344193,NaN,0.773209,023193105,None,None,XSD,2026-08-27
98,CEVA,CEVA INC,None,685601,NaN,0.728430,157210105,None,None,XSD,2026-08-27
99,MRAM,EVERSPIN TECHNOLOGIES INC,None,870627,NaN,0.568289,30041T104,None,None,XSD,2026-08-27
100,NVEC,NVE CORP,None,129606,NaN,0.518832,629445206,None,None,XSD,2026-08-27


In [66]:
%%sql


Exception: Variable Name is not chosen

In [65]:
import plotly
print(plotly.__version__)

7.0.0


In [68]:
import plotly.express as px

fon = "SMH"
d = tum[tum["fund"] == fon].copy()
d["getiri"] = d["ticker"].map(getiri_3ay)

fig = px.treemap(
    d,
    path=[px.Constant(fon), "ticker"],
    values="weight",
    color="getiri",
    color_continuous_scale=["#c0392b", "#f0f0f0", "#27ae60"],
    color_continuous_midpoint=0,
    custom_data=["name", "weight", "getiri"],
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[1]:.1f}%",
    hovertemplate="<b>%{customdata[0]}</b><br>Ağırlık: %{customdata[1]:.2f}%<br>3 aylık: %{customdata[2]:.1f}%<extra></extra>",
)

fig.update_layout(
    title=f"{fon} — portföy dağılımı ve 3 aylık getiri",
    margin=dict(t=50, l=0, r=0, b=0),
    height=600,
)

fig.show()

In [69]:
fonlar = ["SMH", "SOXX", "XSD"]

fon_fiyat = yf.download(
    fonlar + ["GC=F", "^GSPC", "BIL"],
    start="2025-08-01",
    end="2026-08-28",
    auto_adjust=True,
    progress=False,
)["Close"].dropna(how="all")

print(fon_fiyat.shape)
print(fon_fiyat.tail(3).round(2))

(270, 6)
Ticker        BIL    GC=F     SMH    SOXX     XSD    ^GSPC
Date                                                      
2026-08-25  91.62  4638.1  555.82  514.06  490.93  7677.28
2026-08-26  91.63  4598.2  555.77  515.40  490.71  7675.70
2026-08-27  91.63  4609.7  573.00  525.43  500.26  7730.99


In [71]:
def risk_metrikleri(seri, risksiz_gunluk=0.0):
    seri = seri.dropna()
    gunluk = seri.pct_change().dropna()
    fazla = gunluk - risksiz_gunluk

    zirve = seri.cummax()
    dusus = (seri / zirve - 1) * 100

    return pd.Series({
        "getiri": (seri.iloc[-1] / seri.iloc[0] - 1) * 100,
        "volatilite": gunluk.std() * np.sqrt(252) * 100,
        "max_dusus": dusus.min(),
        "sharpe": (fazla.mean() * 252) / (gunluk.std() * np.sqrt(252)),
    })

bil_gunluk = fon_fiyat["BIL"].pct_change().mean()
risk = fon_fiyat.apply(lambda s: risk_metrikleri(s, bil_gunluk)).T
print(risk.round(2))

        getiri  volatilite  max_dusus  sharpe
Ticker                                       
BIL       4.01        0.20      -0.01   -0.00
GC=F     37.70       27.87     -25.06    1.08
SMH     102.42       38.22     -24.62    1.83
SOXX    122.51       44.11     -29.01    1.84
XSD      91.56       45.45     -30.81    1.49
^GSPC    23.93       12.66      -9.10    1.36


In [73]:
import plotly.graph_objects as go

gosterilecek = ["SMH", "SOXX", "XSD", "^GSPC"]
renkler = {"SMH": "#1f77b4", "SOXX": "#ff7f0e", "XSD": "#2ca02c", "^GSPC": "#999999"}

fig = go.Figure()

for t in gosterilecek:
    s = fon_fiyat[t].dropna()
    dusus = (s / s.cummax() - 1) * 100

    fig.add_trace(go.Scatter(
        x=dusus.index,
        y=dusus,
        name="S&P 500" if t == "^GSPC" else t,

        line=dict(color=renkler[t], width=2),
        opacity=0.7,
        hovertemplate="%{x|%d %b %Y}<br>%{y:.1f}%<extra>" + t + "</extra>",
    ))

fig.update_layout(
    title="Zirveden düşüş (drawdown) — son 1 yıl, USD bazlı",
    yaxis_title="Zirveye göre (%)",
    hovermode="x unified",
    height=500,
    margin=dict(t=60, l=60, r=20, b=40),
)

fig.show()

In [74]:
import numpy as np
import pandas as pd


def donem_getirisi(fiyat_df, gun):
    """Son 'gun' işlem günündeki yüzde getiri."""
    p = fiyat_df.iloc[-gun:]
    return ((p.iloc[-1] / p.iloc[0]) - 1) * 100


def yogunlasma(df):
    d = df.sort_values("weight", ascending=False)
    w = d["weight"] / 100
    return {
        "hisse_sayisi": len(d),
        "en_buyuk": d["weight"].iloc[0],
        "top_10": d["weight"].head(10).sum(),
        "hhi": (w ** 2).sum() * 10000,
        "etkin_hisse": 1 / (w ** 2).sum(),
    }


def katki(df, getiri_serisi):
    d = df.copy()
    d["getiri"] = d["ticker"].map(getiri_serisi)
    veri_yok = d[d["getiri"].isna()]["ticker"].tolist()
    d = d.dropna(subset=["getiri"]).copy()
    d["katki"] = (d["weight"] / 100) * d["getiri"]
    return d.sort_values("katki", ascending=False), veri_yok


def risk_metrikleri(seri, risksiz_gunluk=0.0):
    seri = seri.dropna()
    gunluk = seri.pct_change().dropna()
    fazla = gunluk - risksiz_gunluk
    dusus = (seri / seri.cummax() - 1) * 100
    return {
        "getiri": (seri.iloc[-1] / seri.iloc[0] - 1) * 100,
        "volatilite": gunluk.std() * np.sqrt(252) * 100,
        "max_dusus": dusus.min(),
        "sharpe": (fazla.mean() * 252) / (gunluk.std() * np.sqrt(252)),
    }


def drawdown_serisi(seri):
    seri = seri.dropna()
    return (seri / seri.cummax() - 1) * 100

In [80]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
import yfinance as yf

from parsers import parse_vaneck, parse_ishares, parse_spdr
from analytics import (donem_getirisi, yogunlasma, katki,
                       risk_metrikleri, drawdown_serisi)

st.set_page_config(page_title="ETF Analiz", layout="wide")

FONLAR = {
    "Yarı iletken": {
        "SMH":  (parse_vaneck,  "data/raw/SMH_asof_20260827.xlsx"),
        "SOXX": (parse_ishares, "data/raw/SOXX_holdings.csv"),
        "XSD":  (parse_spdr,    "data/raw/holdings-daily-us-en-xsd.xlsx"),
    },
    "Uzay": {},
}

DONEMLER = {"1 ay": 21, "3 ay": 63, "6 ay": 126, "1 yıl": 252}


@st.cache_data
def holdings_yukle(fon, tema):
    parser, yol = FONLAR[tema][fon]
    return parser(yol, fon)


@st.cache_data
def fiyat_yukle(tickerlar):
    df = yf.download(list(tickerlar), start="2025-08-01",
                     auto_adjust=True, progress=False)["Close"]
    return df.dropna(how="all")


# --- Kenar çubuğu ---
st.sidebar.title("ETF Analiz")
tema = st.sidebar.radio("Tema", [t for t in FONLAR if FONLAR[t]])
fon = st.sidebar.radio("Fon", list(FONLAR[tema].keys()))
donem_adi = st.sidebar.selectbox("Dönem", list(DONEMLER.keys()), index=1)
gun = DONEMLER[donem_adi]

st.sidebar.caption("Tüm getiriler USD bazlıdır.")

# --- Veri ---
h = holdings_yukle(fon, tema)
fiyat = fiyat_yukle(tuple(h["ticker"]) + (fon, "^GSPC"))
getiri = donem_getirisi(fiyat, gun)

# --- Başlık ---
st.title(f"{fon} — {donem_adi}")
st.caption(f"Holdings tarihi: {h['date'].iloc[0].date()}")

# --- Kimlik kartı ---
y = yogunlasma(h)
r = risk_metrikleri(fiyat[fon].iloc[-gun:])

k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("Hisse sayısı", y["hisse_sayisi"])
k2.metric("Etkin hisse", f"{y['etkin_hisse']:.1f}")
k3.metric("En büyük pozisyon", f"%{y['en_buyuk']:.1f}")
k4.metric(f"{donem_adi} getiri", f"%{r['getiri']:.1f}")
k5.metric("Maks. düşüş", f"%{r['max_dusus']:.1f}")

st.divider()

# --- Treemap ---
st.subheader("Portföy dağılımı")
d = h.copy()
d["getiri"] = d["ticker"].map(getiri)

fig = px.treemap(
    d, path=[px.Constant(fon), "ticker"], values="weight",
    color="getiri", color_continuous_midpoint=0,
    color_continuous_scale=["#c0392b", "#eeeeee", "#27ae60"],
    custom_data=["name", "weight", "getiri"],
)
fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[1]:.1f}%",
    hovertemplate="<b>%{customdata[0]}</b><br>Ağırlık: %{customdata[1]:.2f}%"
                  f"<br>{donem_adi}: " + "%{customdata[2]:.1f}%<extra></extra>",
)
fig.update_layout(height=500, margin=dict(t=10, l=0, r=0, b=0))
st.plotly_chart(fig, use_container_width=True)

# --- Katkı ---
st.subheader("Getiriye katkı")
kdf, veri_yok = katki(h, getiri)
st.caption("Katkı = güncel ağırlık × dönem getirisi (yaklaşık hesap).")
if veri_yok:
    st.caption(f"Yetersiz veri: {', '.join(veri_yok)}")

ilk_son = pd.concat([kdf.head(8), kdf.tail(8)])
fig2 = px.bar(ilk_son, x="katki", y="ticker", orientation="h",
              color="katki", color_continuous_midpoint=0,
              color_continuous_scale=["#c0392b", "#eeeeee", "#27ae60"])
fig2.update_layout(height=500, yaxis=dict(autorange="reversed"),
                   showlegend=False, coloraxis_showscale=False)
st.plotly_chart(fig2, use_container_width=True)

# --- Drawdown ---
st.subheader("Zirveden düşüş")
fig3 = go.Figure()
for t, ad, renk in [(fon, fon, "#1f77b4"), ("^GSPC", "S&P 500", "#999999")]:
    dd = drawdown_serisi(fiyat[t].iloc[-gun:])
    fig3.add_trace(go.Scatter(x=dd.index, y=dd, name=ad,
                              line=dict(color=renk, width=2)))
fig3.update_layout(height=400, yaxis_title="Zirveye göre (%)",
                   hovermode="x unified", margin=dict(t=10))
st.plotly_chart(fig3, use_container_width=True)

ImportError: cannot import name 'donem_getirisi' from 'analytics' (/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/analytics/__init__.py)

In [81]:
from metrikler import (donem_getirisi, yogunlasma, katki,
                       risk_metrikleri, drawdown_serisi)

ModuleNotFoundError: No module named 'metrikler'

In [ ]:
pip install streamlit
streamlit run app.py

In [ ]:
pip install streamlit
streamlit run app.py